# RPT-1 Client Tutorial (`RPT1Client`)

This notebook shows a DataFrame-first workflow for SAP RPT-1 inferencing:
1. Initialize `RPT1Client` from `.env`
2. `fit(...)` with context rows
3. `predict(...)` on query rows
4. Read predictions as a DataFrame (`PredictionResult.predictions_df`)

The client is inferencing-focused in v1 and expects an existing deployment URL or deployment ID.


## 1) Prerequisites

Install dependencies:

```bash
pip install -r requirements.txt
```

Make sure `.env` contains:

```bash
AICORE_BASE_URL=<...>
AICORE_AUTH_URL=<...>
AICORE_CLIENT_ID=<...>
AICORE_CLIENT_SECRET=<...>
AICORE_RESOURCE_GROUP=default

RPT1_DEPLOYMENT_URL=<...>   # preferred
# or
RPT1_DEPLOYMENT_ID=<...>
```


In [10]:
from pathlib import Path
import json

import pandas as pd

from rpt1_client import (
    RPT1Client,
    RPT1RequestError,
    RPT1ValidationError,
)


## 2) Initialize the client

`from_env()` loads credentials and deployment settings from `.env`.


In [11]:
client = RPT1Client.from_env(
    env_path='.env',
    timeout_seconds=90,
    max_retries=3,
)

print('Client initialized successfully.')
print('Deployment URL set:', bool(client.deployment_url))
print('Deployment ID set :', bool(client.deployment_id))


Client initialized successfully.
Deployment URL set: True
Deployment ID set : True


## 3) Build context and query tables

- Context rows contain known target values.
- Query rows contain the rows you want to predict.


In [12]:
context_df = pd.DataFrame(
    [
        {
            'PRODUCT': 'Office Chair',
            'PRICE': 150.8,
            'ORDERDATE': '02-11-2025',
            'ID': '44',
            'COSTCENTER': 'Office Furniture',
        },
        {
            'PRODUCT': 'Server Rack',
            'PRICE': 2200.0,
            'ORDERDATE': '01-11-2025',
            'ID': '104',
            'COSTCENTER': 'Data Infrastructure',
        },
        {
            'PRODUCT': 'Laptop Stand',
            'PRICE': 49.9,
            'ORDERDATE': '03-11-2025',
            'ID': '88',
            'COSTCENTER': 'Office Furniture',
        },
    ]
)

query_df = pd.DataFrame(
    [
        {
            'PRODUCT': 'Couch',
            'PRICE': 999.99,
            'ORDERDATE': '28-11-2025',
            'ID': '35',
        },
        {
            'PRODUCT': 'Desk Organizer',
            'PRICE': 35.0,
            'ORDERDATE': '30-11-2025',
            'ID': '36',
        },
    ]
)

context_df


,PRODUCT,PRICE,ORDERDATE,ID,COSTCENTER
0,Office Chair,150.8,02-11-2025,44,Office Furniture
1,Server Rack,2200.0,01-11-2025,104,Data Infrastructure
2,Laptop Stand,49.9,03-11-2025,88,Office Furniture


In [13]:
query_df


,PRODUCT,PRICE,ORDERDATE,ID
0,Couch,999.99,28-11-2025,35
1,Desk Organizer,35.00,30-11-2025,36


## 4) Fit the client with context data

This stores the context and prediction configuration (similar to sklearn usage).


In [14]:
client.fit(
    context_df=context_df,
    target_columns=['COSTCENTER'],
    index_column='ID',
    task_types={'COSTCENTER': 'classification'},
)

print('Client fitted.')


Client fitted.


## 5) Predict query rows


In [15]:
try:
    result = client.predict(query_df)
except (RPT1ValidationError, RPT1RequestError) as exc:
    print('Prediction failed:', exc)
    raise

result.predictions_df


,ID,COSTCENTER,COSTCENTER__confidence
0,35,Office Furniture,0.89
1,36,Office Furniture,0.97


## 6) Inspect metadata and raw response


In [17]:
print('Request ID   :', result.request_id)
print('Status code  :', result.status_code)
print('Status msg   :', result.status_message)
print('Metadata     :')
print(json.dumps(result.metadata, indent=2))


Request ID   : c4914b33-3df2-41f1-b163-4fdae0b43ac3
Status code  : 0
Status msg   : ok
Metadata     :
{
  "num_columns": 5,
  "num_predictions": 2,
  "num_query_rows": 2,
  "num_rows": 3
}


In [18]:
# Uncomment to inspect the full response payload
print(json.dumps(result.raw_response, indent=2))


{
  "id": "c4914b33-3df2-41f1-b163-4fdae0b43ac3",
  "metadata": {
    "num_columns": 5,
    "num_predictions": 2,
    "num_query_rows": 2,
    "num_rows": 3
  },
  "predictions": [
    {
      "COSTCENTER": [
        {
          "confidence": 0.89,
          "prediction": "Office Furniture"
        }
      ],
      "ID": "35"
    },
    {
      "COSTCENTER": [
        {
          "confidence": 0.97,
          "prediction": "Office Furniture"
        }
      ],
      "ID": "36"
    }
  ],
  "status": {
    "code": 0,
    "message": "ok"
  }
}


## 7) Optional: Explicit schema

If you already know the column types, pass `data_schema` during `fit(...)`.


In [19]:
explicit_schema = {
    'PRODUCT': {'dtype': 'string'},
    'PRICE': {'dtype': 'numeric'},
    'ORDERDATE': {'dtype': 'date'},
    'ID': {'dtype': 'string'},
    'COSTCENTER': {'dtype': 'string'},
}

client.fit(
    context_df=context_df,
    target_columns=['COSTCENTER'],
    index_column='ID',
    task_types={'COSTCENTER': 'classification'},
    data_schema=explicit_schema,
)

result_with_schema = client.predict(query_df)
result_with_schema.predictions_df


,ID,COSTCENTER,COSTCENTER__confidence
0,35,Office Furniture,0.89
1,36,Office Furniture,0.97


## 8) Mixed task types in one call (classification + regression)



Yes, `RPT1Client` supports multiple target columns with different task types in the same prediction call.



In this example:

- `COSTCENTER` is a **classification** target.

- `DISCOUNT_RATE` is a **regression** target.


In [23]:
context_mixed_df = pd.DataFrame(
    [
        {
            'PRODUCT': 'Office Chair',
            'PRICE': 150.8,
            'QUANTITY': 12,
            'ID': '44',
            'COSTCENTER': 'Office Furniture',
            'DISCOUNT_RATE': 0.08,
        },
        {
            'PRODUCT': 'Server Rack',
            'PRICE': 2200.0,
            'QUANTITY': 2,
            'ID': '104',
            'COSTCENTER': 'Data Infrastructure',
            'DISCOUNT_RATE': 0.15,
        },
        {
            'PRODUCT': 'Desk Organizer',
            'PRICE': 35.0,
            'QUANTITY': 25,
            'ID': '88',
            'COSTCENTER': 'Office Furniture',
            'DISCOUNT_RATE': 0.05,
        },
    ]
)

query_mixed_df = pd.DataFrame(
    [
        {
            'PRODUCT': 'Monitor Arm',
            'PRICE': 210.0,
            'QUANTITY': 8,
            'ID': '501',
        },
        {
            'PRODUCT': 'Network Switch',
            'PRICE': 980.0,
            'QUANTITY': 3,
            'ID': '502',
        },
    ]
)

context_mixed_df


,PRODUCT,PRICE,QUANTITY,ID,COSTCENTER,DISCOUNT_RATE
0,Office Chair,150.8,12,44,Office Furniture,0.08
1,Server Rack,2200.0,2,104,Data Infrastructure,0.15
2,Desk Organizer,35.0,25,88,Office Furniture,0.05


In [24]:
client.fit(
    context_df=context_mixed_df,
    target_columns=['COSTCENTER', 'DISCOUNT_RATE'],
    index_column='ID',
    task_types={
        'COSTCENTER': 'classification',
        'DISCOUNT_RATE': 'regression',
    },
)

mixed_result = client.predict(query_mixed_df)
mixed_result.predictions_df


,ID,COSTCENTER,COSTCENTER__confidence,DISCOUNT_RATE,DISCOUNT_RATE__confidence
0,501,Office Furniture,0.60,0.095965,None
1,502,Data Infrastructure,0.96,0.137921,None


## Troubleshooting

- `401/403`: validate credentials, role collection/scopes, and resource group.
- `404` on `/predict`: deployment URL is wrong, or deployment is not running.
- `422`: check required columns, `target_columns`, and table consistency.
- Weak prediction quality: add more representative context rows and verify data quality.
